# Kidney Patient Data
The objective of this notebook is to generate our dataset for patients with kidney disease. We first define a list of `icd_code` values that correspond to different kidney disease diagnoses. Using this list of `icd_code`s, we then search through *diagnoses_icd.csv* to identify the `subject_id` of patients with some form of kidney disease. To ensure that the computational time required is reasonable, a subset of these `subject_id`s are randomly sampled. For this subset of patients with kidney disease, all lab tests are extracted from *labevents.csv*. The dataset is saved to the file *kidney_disease_random_1000_patients.csv.gz*. Furthermore, for the purpose of having easy access, the `subject_id` values of all patients with kidney disease are saved as *all_subject_ids.json*, while the `subject_id` values of the randomly sampled patients are saved as *sampled_subject_ids.json*.

In [1]:
import pandas as pd
import numpy as np
import json
from random import sample

We first import *d_icd_diagnoses.csv* and *diagnoses_icd.csv* to find the `icd_code` values associated with kidney disease and the diagnoses of patients, respectively. 

In [2]:
d_icd_diagnoses_df = pd.read_csv('../Data/d_icd_diagnoses.csv')
d_icd_diagnoses_df.head()

,icd_code,icd_version,long_title
0,0010,9,Cholera due to vibrio cholerae
1,0011,9,Cholera due to vibrio cholerae el tor
2,0019,9,"Cholera, unspecified"
3,0020,9,Typhoid fever
4,0021,9,Paratyphoid fever A


In [3]:
diagnoses_df = pd.read_csv('../Data/diagnoses_icd.csv')
diagnoses_df.head()

,subject_id,hadm_id,seq_num,icd_code,icd_version
0,10000032,22595853,1,5723,9
1,10000032,22595853,2,78959,9
2,10000032,22595853,3,5715,9
3,10000032,22595853,4,07070,9
4,10000032,22595853,5,496,9


The next cell identifies all patients with some form of kidney disease, then randomly samples 1,000 of those patients for the purpose of our analysis. Notice from the output of the cell that there are 41,386 patients with kidney disease. 

In [4]:
kidney_disease_indices = [2668, 6001, 6002]
kidney_disease_indices += [n for n in range(4660, 4678)]
kidney_disease_indices += [n for n in range(5956, 6000)]
kidney_disease_indices += [n for n in range(6027, 6031)]
kidney_disease_indices += [n for n in range(6086, 6089)]
kidney_disease_indices += [n for n in range(6609, 6614)]
kidney_disease_indices += [n for n in range(37868, 37879)]
kidney_disease_indices = [n - 2 for n in kidney_disease_indices]
icd_codes = list(d_icd_diagnoses_df.iloc[kidney_disease_indices]['icd_code'])
subject_ids = []
for code in icd_codes:
    subject_ids += list(diagnoses_df.loc[diagnoses_df['icd_code'] == code, 'subject_id'])
subject_ids = list(set(subject_ids))
print(len(subject_ids))
sample_ids = sample(subject_ids, 1000)
print(len(sample_ids))


41386
1000


Next, we implement the most computationally expensive part of the notebook. This entails extracting the lab data for each of the 1,000 patients that were randomly sampled above. 

In [11]:
labevents_cols = ['labevent_id',
                 'subject_id',
                 'hadm_id',
                 'specimen_id',
                 'itemid',
                 'order_provider_id',
                 'charttime',
                 'storetime',
                 'value',
                 'valuenum',
                 'valueuom',
                 'ref_range_lower',
                 'ref_range_upper',
                 'flag',
                 'priority',
                 'comments']
kidney_disease_labs_df = pd.DataFrame(columns = labevents_cols,
                                     dtype = object)
chunk_size = 10**6
for n, chunk in enumerate(pd.read_csv('../Data/labevents.csv.gz', compression = 'gzip', chunksize = chunk_size)):
    print(n)
    for patient in sample_ids:
        patient_df = chunk.loc[chunk['subject_id'] == patient]
        kidney_disease_labs_df = pd.concat([kidney_disease_labs_df, patient_df])
        


0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158


Finally, the results from this procedure are saved to the files mentioned at the beginning of this notebook. 

In [12]:
kidney_disease_labs_df.to_csv('../Data/kidney_disease_random_1000_patients.csv.gz',
                             index = False,
                             compression = 'gzip')

In [1]:
with open('../Data/sampled_subject_ids.json', 'w') as file:
    json.dump(sample_ids, file)
with open('../Data/all_subject_ids.json', 'w') as file:
    json.dump(subject_ids, file)

NameError: name 'json' is not defined